# 05 - Gobernanza, Ética y Seguridad de Datos

## Objetivo

En esta etapa se aplicarán principios básicos de gobernanza, privacidad y seguridad sobre el dataset del proyecto EdTech Learning Analytics.

Las actividades principales serán:

- identificar y clasificar los campos sensibles del dataset;
- aplicar técnicas de anonimización o seudonimización;
- definir una política básica de gobernanza de datos;
- elaborar una matriz de riesgos éticos y de seguridad;
- documentar las decisiones tomadas y justificar las técnicas aplicadas.

El objetivo es proteger la información académica y personal de los estudiantes antes de utilizar los datos en análisis avanzados o modelos de inteligencia artificial.

In [33]:
import pandas as pd
from pathlib import Path

In [34]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "datos"
DATA_PROCESSED = DATA_DIR / "procesados"

ruta_dataset_limpio = DATA_PROCESSED / "edtech_dataset_limpio.csv"

## 1. Carga y revisión del dataset

Antes de clasificar los campos según su nivel de sensibilidad, se carga el dataset limpio generado en las etapas anteriores y se revisa su estructura.

Esta validación permite identificar las columnas disponibles y trabajar únicamente con los datos reales del proyecto.

In [35]:
df = pd.read_csv(ruta_dataset_limpio)

print("Dataset cargado correctamente")
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

Dataset cargado correctamente
Número de filas: 200000
Número de columnas: 36


In [36]:
print("Columnas disponibles:\n")

for columna in df.columns:
    print(columna)

Columnas disponibles:

interaction_id
student_id
course_id
timestamp
activity_type
duration_seconds
completion_rate
score
attempts
device_used
help_sought
peer_interaction
age
education_level
study_field
learning_style
device_preference
timezone
previous_online_courses
has_learning_disability
motivation_level
course_title
subject_area
difficulty_level
duration_weeks
instructor_rating
course_format
has_assignments
has_quizzes
has_discussion_forum
certificate_offered
interaction_year
interaction_month
interaction_week
duration_minutes
average_student_score


In [37]:
df.head()

,interaction_id,student_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,...,course_format,has_assignments,has_quizzes,has_discussion_forum,certificate_offered,interaction_year,interaction_month,interaction_week,duration_minutes,average_student_score
0,interaction_000000,student_02020,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,...,Video,1,1,0,1,2024,10,43,6.53,0.7837
1,interaction_000001,student_07410,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,...,Mixed,1,1,1,1,2024,11,46,28.05,0.8064
2,interaction_000002,student_03086,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,...,Text-based,1,1,0,0,2024,6,26,20.25,0.8422
3,interaction_000003,student_05092,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,...,Mixed,0,1,0,1,2024,11,47,4.57,0.7217
4,interaction_000004,student_03626,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,...,Video,1,1,1,1,2024,6,26,13.68,0.8467


## 2. Identificación y clasificación de campos sensibles

Se revisaron las columnas del dataset para determinar qué nivel de protección requiere cada una.

La clasificación utilizada es:

- **Público:** información que puede difundirse sin afectar la privacidad.
- **Interno:** información de uso interno que no es altamente sensible.
- **Confidencial:** información personal, académica o de comportamiento que debe protegerse.
- **Acceso restringido:** información especialmente sensible que requiere controles de acceso más estrictos.

En este proyecto, los campos relacionados con identificación del estudiante, desempeño académico y condiciones personales requieren mayor nivel de protección.

In [38]:
sensibilidad = {
    "student_id": "Acceso restringido",
    "age": "Confidencial",
    "education_level": "Interno",
    "study_field": "Interno",
    "learning_style": "Confidencial",
    "device_preference": "Interno",
    "timezone": "Interno",
    "previous_online_courses": "Interno",
    "has_learning_disability": "Acceso restringido",
    "motivation_level": "Confidencial",

    "interaction_id": "Interno",
    "timestamp": "Confidencial",
    "activity_type": "Interno",
    "duration_seconds": "Interno",
    "completion_rate": "Confidencial",
    "score": "Confidencial",
    "attempts": "Interno",
    "device_used": "Interno",
    "help_sought": "Confidencial",
    "peer_interaction": "Interno",

    "course_id": "Interno",
    "course_title": "Interno",
    "subject_area": "Interno",
    "difficulty_level": "Interno",
    "duration_weeks": "Interno",
    "instructor_rating": "Confidencial",
    "course_format": "Interno",
    "has_assignments": "Interno",
    "has_quizzes": "Interno",
    "has_discussion_forum": "Interno",
    "certificate_offered": "Interno",

    "interaction_year": "Interno",
    "interaction_month": "Interno",
    "interaction_week": "Interno",
    "duration_minutes": "Interno",
    "average_student_score": "Confidencial"
}

In [39]:
for campo, nivel in sensibilidad.items():
    print(f"{campo}: {nivel}")

student_id: Acceso restringido
age: Confidencial
education_level: Interno
study_field: Interno
learning_style: Confidencial
device_preference: Interno
timezone: Interno
previous_online_courses: Interno
has_learning_disability: Acceso restringido
motivation_level: Confidencial
interaction_id: Interno
timestamp: Confidencial
activity_type: Interno
duration_seconds: Interno
completion_rate: Confidencial
score: Confidencial
attempts: Interno
device_used: Interno
help_sought: Confidencial
peer_interaction: Interno
course_id: Interno
course_title: Interno
subject_area: Interno
difficulty_level: Interno
duration_weeks: Interno
instructor_rating: Confidencial
course_format: Interno
has_assignments: Interno
has_quizzes: Interno
has_discussion_forum: Interno
certificate_offered: Interno
interaction_year: Interno
interaction_month: Interno
interaction_week: Interno
duration_minutes: Interno
average_student_score: Confidencial


In [40]:
columnas_sin_clasificar = [
    columna
    for columna in df.columns
    if columna not in sensibilidad
]

print("Columnas sin clasificar:", columnas_sin_clasificar)

Columnas sin clasificar: []


## 3. Técnicas de anonimización y seudonimización

Para proteger la información de los estudiantes se aplicaron dos técnicas:

1. **Seudonimización del identificador del estudiante:** el campo `student_id` se sustituye por un identificador generado mediante una función hash. De esta manera, el estudiante puede seguir siendo distinguido dentro del análisis sin utilizar directamente su identificador original.

2. **Agregación de notas:** las calificaciones individuales se resumen a nivel de curso mediante el promedio de `score`. Esto permite analizar el rendimiento académico general sin exponer directamente las notas de cada estudiante.

Las transformaciones se realizan sobre copias de los datos para conservar el dataset original sin modificaciones.

In [41]:
import hashlib

In [42]:
df_protegido = df.copy()

In [43]:
df_protegido["student_id_hash"] = (
    df_protegido["student_id"]
    .astype(str)
    .apply(
        lambda valor: hashlib.sha256(
            valor.encode()
        ).hexdigest()[:12]
    )
)

In [44]:
df_protegido[
    ["student_id", "student_id_hash"]
].head()

,student_id,student_id_hash
0,student_02020,b260928134ea
1,student_07410,f9c061ec9a42
2,student_03086,6528a9f2b19e
3,student_05092,f08582d4a22b
4,student_03626,ffe2fb94ce9b


In [45]:
df_seudonimizado = df_protegido.drop(
    columns=["student_id"]
)

df_seudonimizado.head()

,interaction_id,course_id,timestamp,activity_type,duration_seconds,completion_rate,score,attempts,device_used,help_sought,...,has_assignments,has_quizzes,has_discussion_forum,certificate_offered,interaction_year,interaction_month,interaction_week,duration_minutes,average_student_score,student_id_hash
0,interaction_000000,course_078,2024-10-24 22:02:09,Note Taking,392,1.0,NaN,1,Tablet,False,...,1,1,0,1,2024,10,43,6.53,0.7837,b260928134ea
1,interaction_000001,course_067,2024-11-15 19:41:09,Assignment Submit,1683,1.0,0.9,2,Mobile,False,...,1,1,1,1,2024,11,46,28.05,0.8064,f9c061ec9a42
2,interaction_000002,course_134,2024-06-29 21:33:09,Resource Download,1215,0.0,NaN,1,Desktop,False,...,1,1,0,0,2024,6,26,20.25,0.8422,6528a9f2b19e
3,interaction_000003,course_112,2024-11-18 03:24:09,Note Taking,274,0.0,NaN,1,Desktop,False,...,0,1,0,1,2024,11,47,4.57,0.7217,f08582d4a22b
4,interaction_000004,course_011,2024-06-30 03:13:09,Note Taking,821,1.0,NaN,1,Mobile,False,...,1,1,1,1,2024,6,26,13.68,0.8467,ffe2fb94ce9b


In [46]:
notas_agregadas = (
    df
    .groupby(
        ["course_id", "course_title"],
        as_index=False
    )
    .agg(
        average_score=("score", "mean"),
        evaluated_activities=("score", "count")
    )
)

In [47]:
notas_agregadas["average_score"] = (
    notas_agregadas["average_score"].round(4)
)

notas_agregadas.head()

,course_id,course_title,average_score,evaluated_activities
0,course_000,Economics 324,0.8061,453
1,course_001,History 563,0.7958,381
2,course_002,History 244,0.8046,440
3,course_003,Computer Science 549,0.8113,460
4,course_004,Chemistry 574,0.8003,472


In [48]:
print(
    "student_id presente en dataset seudonimizado:",
    "student_id" in df_seudonimizado.columns
)

print(
    "student_id_hash presente:",
    "student_id_hash" in df_seudonimizado.columns
)

print(
    "Número de cursos con notas agregadas:",
    len(notas_agregadas)
)

student_id presente en dataset seudonimizado: False
student_id_hash presente: True
Número de cursos con notas agregadas: 150


## 4. Política básica de gobernanza de datos

Se define una política básica para establecer cómo deben utilizarse, protegerse y conservarse los datos del proyecto EdTech Learning Analytics.

### 4.1 Propósito

Establecer lineamientos para el uso responsable de la información académica y de comportamiento de los estudiantes, garantizando que los datos sean utilizados únicamente para fines de análisis educativo, mejora del aprendizaje y desarrollo de modelos de analítica o inteligencia artificial.

### 4.2 Alcance

La política aplica a los datos relacionados con:

- estudiantes;
- interacciones dentro de la plataforma educativa;
- resultados académicos;
- cursos;
- métricas de participación y comportamiento.

### 4.3 Uso de los datos

Los datos deberán utilizarse únicamente para fines académicos, analíticos y de investigación relacionados con el proyecto.

No deberán utilizarse para propósitos diferentes a los definidos sin una revisión previa del uso propuesto.

Cuando sea posible, los análisis deberán realizarse utilizando datos seudonimizados o agregados para reducir la exposición de información sensible.

### 4.4 Control de acceso

El acceso a los datos deberá seguir el principio de privilegio mínimo.

Los niveles de acceso considerados son:

- **Datos internos:** disponibles para los miembros autorizados del equipo del proyecto.
- **Datos confidenciales:** accesibles únicamente cuando sean necesarios para el análisis.
- **Datos de acceso restringido:** disponibles únicamente para responsables autorizados y preferentemente mediante versiones seudonimizadas.

Los identificadores directos o campos especialmente sensibles no deberán ser utilizados cuando no sean necesarios para el objetivo del análisis.

### 4.5 Retención de datos

Los datos deberán conservarse únicamente durante el tiempo necesario para completar los objetivos académicos y analíticos del proyecto.

Al finalizar el periodo de uso:

- los datasets temporales deberán eliminarse;
- las versiones con información sensible deberán conservarse únicamente si existe una necesidad justificada;
- para análisis posteriores deberán preferirse versiones anonimizadas, seudonimizadas o agregadas.

### 4.6 Protección de los datos

Los datos sensibles deberán protegerse mediante técnicas como:

- seudonimización;
- agregación;
- control de acceso;
- reducción de información innecesaria.

En este proyecto se aplicó seudonimización al identificador del estudiante y agregación de las calificaciones por curso.

### 4.7 Responsabilidades

Para efectos del proyecto se consideran los siguientes roles:

- **Propietario de los datos:** responsable de definir el uso permitido de la información.
- **Responsable de datos:** encargado de aplicar controles de calidad, protección y acceso.
- **Analistas:** usuarios autorizados para realizar análisis utilizando únicamente los datos necesarios para su función.

### 4.8 Revisión

La política deberá revisarse cuando:

- se incorporen nuevas fuentes de datos;
- se agreguen nuevas variables sensibles;
- cambie el propósito del análisis;
- se desarrollen modelos de inteligencia artificial que utilicen información de estudiantes.

## 5. Matriz de riesgos éticos y de seguridad

Se identificaron los principales riesgos asociados al uso de datos académicos y de comportamiento de estudiantes dentro del proyecto EdTech Learning Analytics.

La matriz considera:

- **Riesgo:** situación que puede afectar la privacidad, seguridad o uso ético de los datos.
- **Probabilidad:** posibilidad de que el riesgo ocurra.
- **Impacto:** nivel de afectación potencial.
- **Control:** medida utilizada para reducir el riesgo.
- **Responsable:** rol encargado de aplicar o supervisar el control.

| Riesgo | Probabilidad | Impacto | Control | Responsable |
|---|---|---|---|---|
| Exposición del identificador del estudiante | Media | Alto | Seudonimización de `student_id` y restricción de acceso | Responsable de datos |
| Acceso no autorizado a información académica | Media | Alto | Control de acceso basado en roles y privilegio mínimo | Responsable de datos |
| Divulgación de información sobre discapacidad de aprendizaje | Baja | Alto | Clasificación como acceso restringido y uso únicamente cuando sea necesario | Propietario de datos |
| Uso indebido de calificaciones individuales | Media | Alto | Uso de datos agregados para análisis generales | Analistas / Responsable de datos |
| Reidentificación de estudiantes mediante combinación de variables | Baja | Alto | Reducción de variables sensibles y uso de datos seudonimizados | Responsable de datos |
| Sesgo en futuros modelos de IA | Media | Alto | Revisión de variables, calidad de datos y evaluación de resultados por grupos | Científico o analista de datos |
| Uso de los datos para fines distintos al objetivo académico | Baja | Medio | Política de uso y autorización previa para nuevos propósitos | Propietario de datos |
| Conservación innecesaria de datos sensibles | Media | Medio | Política de retención y eliminación de datasets temporales | Responsable de datos |
| Interpretación incorrecta de métricas de comportamiento | Media | Medio | Documentar el significado y limitaciones de las variables | Analistas |

## 6. Documentación y justificación de decisiones

Durante esta etapa se tomaron decisiones orientadas a proteger la privacidad de los estudiantes y reducir la exposición innecesaria de información sensible.

### 6.1 Clasificación de campos

Los campos del dataset fueron clasificados de acuerdo con su nivel de sensibilidad.

Se consideraron especialmente sensibles:

- `student_id`, por permitir identificar de forma única a un estudiante;
- `has_learning_disability`, por representar una condición personal sensible;
- `score` y `average_student_score`, por contener información de desempeño académico;
- `completion_rate`, `help_sought` y `motivation_level`, por reflejar comportamiento y desempeño individual;
- `timestamp`, por permitir analizar patrones de actividad asociados a un estudiante.

La clasificación permite definir controles de acceso distintos según el nivel de sensibilidad de cada variable.

### 6.2 Seudonimización de `student_id`

Se aplicó una función hash SHA-256 sobre `student_id` para generar un identificador alternativo.

Esta técnica fue seleccionada porque permite mantener la relación entre las interacciones pertenecientes al mismo estudiante sin utilizar directamente su identificador original.

Posteriormente, el campo original `student_id` puede eliminarse de las versiones utilizadas para análisis cuando no sea necesario conservarlo.

Esta técnica reduce el riesgo de exposición directa de la identidad del estudiante.

### 6.3 Agregación de calificaciones

Como segunda técnica de protección se calculó el promedio de las calificaciones a nivel de curso.

La agregación permite analizar el rendimiento general de los cursos sin mostrar las calificaciones individuales de cada estudiante.

Esta decisión resulta adecuada para análisis generales donde no es necesario identificar el desempeño específico de una persona.

### 6.4 Conservación del dataset original

Las técnicas de protección se aplicaron sobre copias del DataFrame.

El dataset original no fue modificado directamente.

Esta decisión permite mantener trazabilidad entre los datos originales y las versiones utilizadas para análisis, evitando alteraciones irreversibles durante las pruebas.

### 6.5 Control de acceso

Se estableció el principio de privilegio mínimo como criterio para el acceso a los datos.

Los usuarios deberían acceder únicamente a las variables necesarias para desarrollar su función.

Los campos clasificados como confidenciales o de acceso restringido requieren controles adicionales y no deberían utilizarse cuando el análisis pueda realizarse con información agregada o seudonimizada.

### 6.6 Retención de datos

Se definió que los datos deben conservarse únicamente durante el periodo necesario para cumplir los objetivos académicos y analíticos del proyecto.

Las copias temporales y datasets con información sensible deben eliminarse cuando dejen de ser necesarios.

Para análisis posteriores se recomienda utilizar preferentemente versiones seudonimizadas o agregadas.

### 6.7 Consideraciones éticas

El uso de variables relacionadas con desempeño, comportamiento o condiciones personales puede afectar la interpretación de los resultados.

Por esta razón:

- las métricas académicas no deben utilizarse de forma aislada para emitir conclusiones sobre un estudiante;
- las variables sensibles deben utilizarse únicamente cuando exista una finalidad justificada;
- los futuros modelos de inteligencia artificial deberán revisarse para detectar posibles sesgos;
- cualquier clasificación de estudiantes deberá interpretarse como apoyo al análisis y no como una decisión definitiva sobre su desempeño o capacidad.

### 6.8 Decisión general

Las técnicas seleccionadas buscan mantener un equilibrio entre la utilidad analítica del dataset y la protección de la información.

La seudonimización permite continuar analizando patrones individuales sin utilizar directamente el identificador original, mientras que la agregación permite realizar análisis generales reduciendo la exposición de información académica individual.

Estas decisiones proporcionan una base de gobernanza para continuar utilizando el dataset en futuras etapas de analítica e inteligencia artificial.

## 7. Conclusión

En esta etapa se incorporaron principios de gobernanza, privacidad, ética y seguridad al proyecto EdTech Learning Analytics.

Se identificaron y clasificaron los campos del dataset según su nivel de sensibilidad, prestando especial atención a los identificadores de estudiantes, variables académicas, métricas de comportamiento y datos personales sensibles.

También se aplicaron dos técnicas de protección de datos: la seudonimización del identificador del estudiante mediante una función hash y la agregación de calificaciones por curso. Estas técnicas permiten mantener la utilidad analítica de la información reduciendo la exposición directa de datos individuales.

Además, se definió una política básica de gobernanza que establece lineamientos de uso, acceso, protección y retención de los datos. Como complemento, se elaboró una matriz de riesgos éticos y de seguridad para identificar posibles escenarios de exposición, acceso no autorizado, reidentificación, uso indebido de información académica y sesgo en futuros modelos de inteligencia artificial.

Finalmente, se documentaron las decisiones tomadas y la justificación de las técnicas aplicadas.

Con esta etapa, el proyecto cuenta con una base de gobernanza que permite continuar utilizando los datos de forma más responsable, controlada y adecuada para futuras actividades de analítica avanzada e inteligencia artificial.